# Sprint 5 Runner (Colab)

<!-- Cell 0: Judul notebook - intro Sprint 5 Plan v3 -->

Notebook khusus untuk menjalankan `Sprint 5 Plan v3` end-to-end dengan output log tampil penuh di setiap sel.

## 0) Settings

<!-- Cell 1: Header section Settings -->

Isi `REPO_URL` dengan repository kamu. Kalau repo private, pastikan token/Git auth sudah siap di Colab.

In [1]:
# Cell 2: Core settings - REPO_URL, branch, path, stage toggles
REPO_URL = 'https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git'  # contoh: https://github.com/<user>/<repo>.git
REPO_BRANCH = 'feat/sprint5-cse-fpr-reduction'  # branch yang mau dipakai di Colab
PROJECT_NAME = 'nids-cnn-lstm-autoencoder'
DRIVE_ROOT = '/content/drive/MyDrive/nids-cnn-lstm-autoencoder'

# Raw dataset source (default mengikuti Sprint 3)
# Jika folder ini tidak ada, notebook fallback ke {DRIVE_ROOT}/data/raw
RAW_DRIVE_SOURCE = '/content/drive/MyDrive/nids-data/raw'

FORCE_RECLONE = False

# Stage toggles (set True/False sesuai kebutuhan)
RUN_STAGE0 = True
RUN_STAGE1 = True
RUN_STAGE2 = False
RUN_STAGE3_4 = True
RUN_SUMMARIZE = True


In [2]:
# Cell 3: Colab bootstrap + mount Google Drive
import os
import sys
import json
import time
import shlex
import shutil
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


IN_COLAB = True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT = /content/drive/MyDrive/nids-cnn-lstm-autoencoder


In [3]:
# Cell 4: Clone atau update repo di /content + checkout branch
PROJECT_ROOT = Path('/content') / PROJECT_NAME

if FORCE_RECLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    if '<REPO_URL_HERE>' in REPO_URL:
        raise ValueError('Set REPO_URL dulu di cell Settings')
    cmd = ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())

# Always sync and checkout selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all', '--prune'], check=False)

# Try checkout branch; if branch only exists on origin, create tracking local branch.
ret = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_BRANCH], check=False)
if ret.returncode != 0:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '-b', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)

# Pull latest on selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', REPO_BRANCH], check=False)

active_branch = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], text=True).strip()
print('[GIT] active branch =', active_branch)


PROJECT_ROOT = /content/nids-cnn-lstm-autoencoder
cwd = /content/nids-cnn-lstm-autoencoder
[GIT] active branch = feat/sprint5-cse-fpr-reduction


In [4]:
# Cell 5: Install dependencies (Colab-safe, skip reinstall numpy/tensorflow)
from pathlib import Path
import importlib.util

req = Path('requirements.txt').resolve()
if not req.exists():
    raise FileNotFoundError(f'requirements.txt not found at {req}')


def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)


if IN_COLAB:
    # IMPORTANT:
    # Colab sudah punya numpy/pandas/scikit/tensorflow yang saling kompatibel.
    # Reinstall paket-paket ini sering memicu ABI mismatch (numpy.dtype size changed).
    print('[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).')

    # Install only lightweight utilities if missing.
    lightweight = [
        'pyyaml',
        'joblib',
        'seaborn',
    ]

    for pkg in lightweight:
        mod = 'yaml' if pkg == 'pyyaml' else pkg
        if importlib.util.find_spec(mod) is None:
            _run([sys.executable, '-m', 'pip', 'install', pkg])
        else:
            print(f'[INFO] {pkg} already available')

    print('\n[OK] Dependency step finished (Colab-safe mode).')
    print('[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.')
else:
    # Local/non-Colab: follow project requirements as usual.
    _run([sys.executable, '-m', 'pip', 'install', '-r', str(req)])


[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).
[INFO] pyyaml already available
[INFO] joblib already available
[INFO] seaborn already available

[OK] Dependency step finished (Colab-safe mode).
[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.


In [5]:
# Cell 6: Link data/model/results sprint5 ke Drive (persist setelah disconnect)
os.chdir(PROJECT_ROOT)

raw_source = RAW_DRIVE_SOURCE if Path(RAW_DRIVE_SOURCE).exists() else f'{DRIVE_ROOT}/data/raw'
print(f'[RAW] using source: {raw_source}')

paths = [
    ('data/raw', raw_source),
    ('data/sprint5', f'{DRIVE_ROOT}/data/sprint5'),
    ('models/sprint5', f'{DRIVE_ROOT}/models/sprint5'),
    ('results/sprint5', f'{DRIVE_ROOT}/results/sprint5'),
]

for _, dst in paths:
    Path(dst).mkdir(parents=True, exist_ok=True)

for src, dst in paths:
    src_path = Path(src)
    if src_path.is_symlink() or src_path.exists():
        if src_path.is_symlink() or src_path.is_file():
            src_path.unlink()
        else:
            shutil.rmtree(src_path)
    src_path.parent.mkdir(parents=True, exist_ok=True)
    src_path.symlink_to(Path(dst), target_is_directory=True)
    print(f'[LINK] {src} -> {dst}')

print('[OK] Symlink setup complete')

# Fast preflight check
required_dirs = [
    Path('data/raw/CIC-IDS2017'),
    Path('data/raw/CSE-CIC-IDS2018'),
]
for d in required_dirs:
    print(f'[CHECK] {d}:', 'OK' if d.exists() else 'MISSING')


[RAW] using source: /content/drive/MyDrive/nids-data/raw
[LINK] data/raw -> /content/drive/MyDrive/nids-data/raw
[LINK] data/sprint5 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/data/sprint5
[LINK] models/sprint5 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/models/sprint5
[LINK] results/sprint5 -> /content/drive/MyDrive/nids-cnn-lstm-autoencoder/results/sprint5
[OK] Symlink setup complete
[CHECK] data/raw/CIC-IDS2017: OK
[CHECK] data/raw/CSE-CIC-IDS2018: OK


In [6]:
# Cell 7: GPU check + run_cmd_stream helper (streaming output ke cell)
try:
    import tensorflow as tf
except Exception as e:
    print('[ERROR] TensorFlow import gagal:', repr(e))
    print('Kemungkinan besar environment ABI belum sinkron setelah pip install.')
    print('Solusi: Runtime > Restart runtime, lalu jalankan lagi dari cell GPU check ini.')
    raise

print('Python executable :', sys.executable)
print('TensorFlow        :', tf.__version__)
print('Built with CUDA   :', tf.test.is_built_with_cuda())
print('Built with GPU sup:', tf.test.is_built_with_gpu_support())
print('Physical GPU list :', tf.config.list_physical_devices('GPU'))
print('Logical GPU list  :', tf.config.list_logical_devices('GPU'))

# Maksimalkan penggunaan GPU: memory growth agar TF pakai memori dinamis
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
if gpus:
    print('[GPU] Memory growth enabled -> training/eval akan memaksimalkan GPU.')


def _fmt_duration(seconds: float) -> str:
    sec = max(0, int(seconds))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h}h {m:02d}m {s:02d}s'
    if m > 0:
        return f'{m}m {s:02d}s'
    return f'{s}s'


def _normalize_cmd(cmd: str):
    parts = shlex.split(cmd, posix=False)
    if parts and parts[0].lower() == 'python':
        parts[0] = sys.executable
    return parts


def run_cmd_stream(title: str, cmd: str, log_to_file: bool = True):
    """Jalankan command dengan streaming output ke cell. Log lengkap disimpan ke file."""
    print(f'[RUN] {title}', flush=True)
    args = _normalize_cmd(cmd)
    print('[CMD]', ' '.join(args), flush=True)
    t0 = time.time()

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        args,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    log_path = None
    log_file = None
    if log_to_file:
        log_dir = PROJECT_ROOT / 'results' / 'sprint5' / 'runtime'
        log_dir.mkdir(parents=True, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
        log_path = log_dir / f'log_{title.replace(" ", "_")}_{ts}.txt'
        log_file = open(log_path, 'w', encoding='utf-8')

    last_line = ''
    try:
        for line in proc.stdout or []:
            print(line, end='', flush=True)
            if log_file:
                log_file.write(line)
                log_file.flush()
            last_line = line.strip()
    finally:
        if log_file:
            log_file.close()
            print(f'\n[LOG] Full log saved to: {log_path}', flush=True)

    ret = proc.wait()
    print(f'\n[EXIT CODE] {ret}', flush=True)
    if ret != 0:
        raise RuntimeError(f"{title} failed (exit={ret}). Last line: {last_line}")

    print(f'[DONE] {title} in {_fmt_duration(time.time() - t0)}', flush=True)


def run_stage(stage_name: str):
    run_cmd_stream(
        title=f'Sprint5 {stage_name}',
        cmd=f'python scripts/sprint5/research_runner.py --stage-names {stage_name} --skip-existing',
    )


Python executable : /usr/bin/python3
TensorFlow        : 2.19.0
Built with CUDA   : True
Built with GPU sup: True
Physical GPU list : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPU list  : [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
[GPU] Memory growth enabled -> training/eval akan memaksimalkan GPU.


## 1) Optional Dry-Run

<!-- Cell 8: Header dry-run - validasi tanpa eksekusi real -->

Validasi command tanpa eksekusi real training/eval.

In [7]:
# Cell 9: Execute dry-run stage0 (validasi command tanpa training/eval real)
run_cmd_stream(
    title='Sprint5 Dry-Run Stage0',
    cmd='python scripts/sprint5/research_runner.py --dry-run --stage-names stage0 --no-summarize',
)


[RUN] Sprint5 Dry-Run Stage0
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --dry-run --stage-names stage0 --no-summarize
[RUN] s5_00_lock_baseline | stage=stage0 | stages=['preprocess', 'train', 'eval'] | tag=s5_00_lock_baseline
[CMD] /usr/bin/python3 scripts/sprint5/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_00_lock_baseline.yaml
[CMD] /usr/bin/python3 scripts/sprint5/train.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_00_lock_baseline.yaml --variant hybrid
[CMD] /usr/bin/python3 scripts/sprint5/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_00_lock_baseline.yaml --model models/sprint5/s5_00_lock_baseline/cnn_lstm_ae/best_model.keras --tag s5_00_lock_baseline
[RUN] s5_m07_rerun | stage=stage0 | stages=['train', 'eval'] | tag=s5_m07_rerun
[CMD] /usr/bin/python3 scripts/sprint5/train.py --config /content/nids-cnn-lstm-autoencoder/research/spr

## 2) Execute Sprint 5 Stages

<!-- Cell 10: Header execute stages -->

In [8]:
# Cell 11: Execute stage0 (repro + strict contract runs)
if RUN_STAGE0:
    run_stage('stage0')
else:
    print('[SKIP] stage0')


[RUN] Sprint5 stage0
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --stage-names stage0 --skip-existing
[SKIP] s5_00_lock_baseline metrics already exist for tag=s5_00_lock_baseline
[SKIP] s5_m07_rerun metrics already exist for tag=s5_m07_rerun
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json

[LOG] Full log saved to: /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/log_Sprint5_stage0_20260301_092544.txt

[EXIT CODE] 0
[DONE] Sprint5 stage0 in 5s


In [9]:
# Cell 12: Execute stage1 (7 target threshold variant runs)
if RUN_STAGE1:
    run_stage('stage1')
else:
    print('[SKIP] stage1')


[RUN] Sprint5 stage1
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --stage-names stage1 --skip-existing
[RUN] s5_t01_target_p95 | stage=stage1 | stages=['eval'] | tag=s5_t01_target_p95
[CMD] /usr/bin/python3 scripts/sprint5/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_t01_target_p95.yaml --model models/sprint5/s5_00_lock_baseline/cnn_lstm_ae/best_model.keras --tag s5_t01_target_p95
2026-03-01 09:25:52.634557: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772357152.656354    8708 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772357152.663640    8708 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regist

In [10]:
# Cell 13: Execute stage2 (Sprint 5: no stage2, skip)
if RUN_STAGE2:
    run_stage('stage2')
else:
    print('[SKIP] stage2')


[SKIP] stage2


In [11]:
# Cell 14: Execute stage3 + stage4 (guardrail relax + seed candidates)
if RUN_STAGE3_4:
    run_cmd_stream(
        title='Sprint5 stage3+stage4',
        cmd='python scripts/sprint5/research_runner.py --stage-names stage3,stage4 --skip-existing',
    )
else:
    print('[SKIP] stage3,stage4')


[RUN] Sprint5 stage3+stage4
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --stage-names stage3,stage4 --skip-existing
[RUN] s5_h01_guardrail_fpr012 | stage=stage3 | stages=['eval'] | tag=s5_h01_guardrail_fpr012
[CMD] /usr/bin/python3 scripts/sprint5/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_h01_guardrail_fpr012.yaml --model /content/sprint5_lock/s5_h01_guardrail_fpr012/best_model.keras --tag s5_h01_guardrail_fpr012
2026-03-01 09:51:48.583421: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772358708.606016   34972 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772358708.613958   34972 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cu

In [12]:
# Cell 15: Execute summarize-only (aggregate hasil, gate decision, report)
if RUN_SUMMARIZE:
    run_cmd_stream(
        title='Sprint5 summarize-only',
        cmd='python scripts/sprint5/research_runner.py --summarize-only',
    )
else:
    print('[SKIP] summarize-only')


[RUN] Sprint5 summarize-only
[CMD] /usr/bin/python3 scripts/sprint5/research_runner.py --summarize-only
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json

[LOG] Full log saved to: /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/log_Sprint5_summarize-only_20260301_103120.txt

[EXIT CODE] 0
[DONE] Sprint5 summarize-only in 0s


## 3) Show Outputs (Summary, Gate, Report)

<!-- Cell 16: Header show outputs -->

In [13]:
# Cell 17: Tampilkan summary.csv, gate_decision.json, report markdown
import pandas as pd
from IPython.display import display, Markdown

summary_path = PROJECT_ROOT / 'results/sprint5/summary.csv'
gate_path = PROJECT_ROOT / 'results/sprint5/gate_decision.json'
report_path = PROJECT_ROOT / 'docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md'
status_path = PROJECT_ROOT / 'results/sprint5/runtime/run_status.json'

print('summary_path =', summary_path)
print('gate_path    =', gate_path)
print('report_path  =', report_path)
print('status_path  =', status_path)

if summary_path.exists():
    df = pd.read_csv(summary_path)
    print('\n[SUMMARY HEAD]')
    display(df.head(30))
    print('\n[STATUS COUNTS]')
    if 'terminal_status' in df.columns:
        display(df['terminal_status'].value_counts(dropna=False))
else:
    print('[WARN] summary.csv not found')

if gate_path.exists():
    gate = json.loads(gate_path.read_text(encoding='utf-8'))
    print('\n[GATE DECISION]')
    print(json.dumps(gate, indent=2))
else:
    print('[WARN] gate_decision.json not found')

if report_path.exists():
    print('\n[REPORT PREVIEW]')
    txt = report_path.read_text(encoding='utf-8')
    display(Markdown(txt[:8000]))
else:
    print('[WARN] report markdown not found')

if status_path.exists():
    print('\n[RUNTIME STATUS JSON]')
    print(status_path.read_text(encoding='utf-8')[:8000])


summary_path = /content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv
gate_path    = /content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json
report_path  = /content/nids-cnn-lstm-autoencoder/docs/sprint5/RESEARCH_REPORT_CSE_F1_SPRINT5.md
status_path  = /content/nids-cnn-lstm-autoencoder/results/sprint5/runtime/run_status.json

[SUMMARY HEAD]


,run_id,stage_name,profile,model_variant,active,condition,stages,tag,terminal_status,retry_count,...,cse_prec,cse_rec,cse_f1,cse_fpr,cse_auc,f1_gap,auc_gap,accuracy_gap,wall_time_sec,last_error
0,s5_00_lock_baseline,stage0,s5_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s5_00_lock_baseline,success,0,...,0.874147,0.625923,0.729498,0.223683,NaN,0.102882,0.0,0.098824,2030.676788,NaN
1,s5_m07_rerun,stage0,s5_hybrid_zero_shot_anomaly,hybrid,True,always,"train,eval",s5_m07_rerun,success,0,...,0.885812,0.698368,0.781001,0.223459,NaN,0.046242,0.0,0.040696,1895.925668,NaN
2,s5_t01_target_p95,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t01_target_p95,success,0,...,0.818296,0.090718,0.163330,0.050002,NaN,0.615469,0.0,0.376852,148.953151,NaN
3,s5_t02_target_p97,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t02_target_p97,success,0,...,0.786038,0.044404,0.084059,0.030002,NaN,0.688204,0.0,0.397703,49.648651,NaN
4,s5_t03_target_p98,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t03_target_p98,success,0,...,0.655400,0.015324,0.029949,0.020000,NaN,0.721921,0.0,0.394922,49.868047,NaN
5,s5_t04_target_p99,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t04_target_p99,success,0,...,0.622131,0.006634,0.013128,0.010002,NaN,0.669943,0.0,0.332463,49.653070,NaN
6,s5_t05_target_p97_sub5pct,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t05_target_p97_sub5pct,success,0,...,0.777619,0.039861,0.075835,0.028296,NaN,0.695194,0.0,0.399221,50.199284,NaN
7,s5_t06_target_p97_sub10pct,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t06_target_p97_sub10pct,success,0,...,0.780241,0.041696,0.079162,0.029151,NaN,0.692467,0.0,0.398749,49.663279,NaN
8,s5_t07_source_calib_guardrail,stage1,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_t07_source_calib_guardrail,success,0,...,0.874147,0.625923,0.729498,0.223683,NaN,0.102882,0.0,0.098824,1155.430669,NaN
9,s5_h01_guardrail_fpr012,stage3,s5_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s5_h01_guardrail_fpr012,success,0,...,0.870093,0.650106,0.744182,0.240929,NaN,0.098034,0.0,0.097272,1186.534233,NaN



[STATUS COUNTS]


,count
terminal_status,
success,11
skipped_gate,2



[GATE DECISION]
{
  "generated_at": "2026-03-01T10:31:20.773978",
  "gate_pass": false,
  "reason": "failed_constraints",
  "adaptive_recall_target": 0.75,
  "adaptive_target_reasoning": "max(0.7500, baseline(0.0907)+0.0000)",
  "best_stage3_run_id": "s5_h02_guardrail_fpr015",
  "best_stage3_metrics": {
    "cse_recall": 0.692531678521269,
    "cse_precision": 0.8650786851154477,
    "cse_f1": 0.7692480839138802,
    "cic_fpr": 0.1465864080092318,
    "threshold": 3.495636224746704
  },
  "pivot_recommendation": "USAD",
  "stage_validity": {
    "stage1": {
      "required": 7,
      "valid_count": 7,
      "passed": true,
      "status": "ok"
    },
    "stage2": {
      "required": 0,
      "valid_count": 0,
      "passed": true,
      "status": "ok"
    },
    "stage3": {
      "required": 2,
      "valid_count": 2,
      "passed": true,
      "status": "ok"
    },
    "stage4": {
      "required": 2,
      "valid_count": 0,
      "passed": true,
      "status": "skipped_by_design"

# RESEARCH REPORT CSE F1 - SPRINT 4

- generated_at: 2026-03-01T10:31:20.777780
- source_summary: `/content/nids-cnn-lstm-autoencoder/results/sprint5/summary.csv`
- gate_decision: `/content/nids-cnn-lstm-autoencoder/results/sprint5/gate_decision.json`

## Ringkasan / Summary

- Objective: maximize CSE recall dengan guardrail CIC FPR rendah.
- Objective (EN): maximize CSE recall under low CIC-FPR guardrail.
- Gate pass: `False` (reason: `failed_constraints`).
- Adaptive recall target: `0.75` (max(0.7500, baseline(0.0907)+0.0000)).

## Stage Validity

| stage | required_valid_runs | valid_runs | status |
| --- | ---: | ---: | --- |
| stage1 | 7 | 7 | ok |
| stage2 | 0 | 0 | ok |
| stage3 | 2 | 2 | ok |
| stage4 | 2 | 0 | skipped_by_design |

## Best Candidate Per Stage

| stage | run_id | mode | cse_recall | cse_precision | cse_f1 | cic_fpr |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| stage0 | s5_m07_rerun | fallback | 0.6984 | 0.8858 | 0.7810 | 0.1087 |
| stage1 | s5_t01_target_p95 | hard | 0.0907 | 0.8183 | 0.1633 | 0.0108 |
| stage2 | - | - | - | - | - | - |
| stage3 | s5_h02_guardrail_fpr015 | worst_case | 0.6925 | 0.8651 | 0.7692 | 0.1466 |
| stage4 | - | - | - | - | - | - |

## Operational Notes

- ID Primary: Terminologi teknis tetap English untuk konsistensi.
- EN Mirror: Technical keywords are intentionally kept in English.

## Run Status

- total_runs: 13
- success: 11
- failed_experiment: 0
- failed_infra_exhausted: 0
- pending_or_skipped: 2



[RUNTIME STATUS JSON]
{
  "s5_00_lock_baseline": {
    "retry_count": 0,
    "terminal_status": "success",
    "last_error": "",
    "hash_violation": false,
    "wall_time_sec": 2030.6767876148224,
    "attempt_logs": []
  },
  "s5_m07_rerun": {
    "retry_count": 0,
    "terminal_status": "success",
    "last_error": "",
    "hash_violation": false,
    "wall_time_sec": 1895.9256675243378,
    "attempt_logs": []
  },
  "s5_t01_target_p95": {
    "retry_count": 0,
    "terminal_status": "success",
    "last_error": "",
    "hash_violation": false,
    "wall_time_sec": 148.9531512260437,
    "attempt_logs": [
      {
        "ts": "2026-03-01T07:22:39.140806",
        "kind": "experiment",
        "error": "Command failed rc=1 after 13.30s: /usr/bin/python3 scripts/sprint5/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint5/generated_configs/s5_t01_target_p95.yaml --model models/sprint5/s5_00_lock_baseline/cnn_lstm_ae/best_model.keras --tag s5_t01_target_p95"
      }


## 4) Resume Guide

<!-- Cell 18: Panduan resume setelah runtime putus -->

Kalau runtime Colab putus:
1. Run lagi dari cell mount + clone/pull + symlink.
2. Jalankan stage berikutnya atau stage yang sama.
3. `--skip-existing` akan melanjutkan dari artifact yang sudah ada.

## 5) Analisis Hasil Run — Sprint 5 Full Run History

> Bagian ini berisi temuan mendalam dari eksekusi Sprint 5 yang telah dijalankan di Colab (Tesla T4, TF 2.19.0).
> Diisi otomatis dari analisis output `histories-do-not-edit/sprint5-full-run.ipynb`.

---

### Ringkasan Status Run (13 total)

| Status | Jumlah | Keterangan |
|---|---|---|
| `success` | 2 | Stage 0 saja (lock_baseline + m07_rerun) |
| `failed_experiment` | 7 | Seluruh Stage 1 — FileNotFoundError |
| `skipped_inconclusive` | 2 | Stage 3 — bergantung stage1 yang gagal |
| `skipped_gate` | 2 | Stage 4 — gate_pass = False |

### Metrik Stage 0 yang Berhasil

| Run | CSE Recall | CSE F1 | CSE FPR | F1 Gap | Wall Time |
|---|---|---|---|---|---|
| `s5_00_lock_baseline` | 0.626 | 0.729 | 0.224 | 0.103 | ~33m 50s |
| `s5_m07_rerun` (mse_mae_mix α=0.7) | **0.698** | **0.781** | 0.224 | **0.046** | ~31m 36s |

**Temuan:** `mse_mae_mix` loss secara konsisten menghasilkan F1 lebih tinggi (+5.2pp) dan F1 gap jauh lebih kecil (0.046 vs 0.103), artinya generalisasi domain lebih baik.

### Gate Decision
```
gate_pass: false
reason: stage3_no_candidate
adaptive_recall_target: 0.75  (fixed_floor_only — no prior baseline)
pivot_recommendation: USAD
count_under_guardrail: 0  (CSE FPR ~0.224 >> guardrail 0.10)
```

### Root Cause — Bug Kritis Stage 1

**Error:**
```
FileNotFoundError: 'data/sprint5/s5_m07_rerun/processed/cic_val.npz'
```

**Rantai masalah:**
1. Semua 7 run Stage 1 memakai `reuse_data_from_best_stage: stage0`.
2. `select_best_run_id_for_stage('stage0')` memilih **`s5_m07_rerun`** (recall=0.698 > lock_baseline 0.626).
3. `apply_data_reuse_inputs(cfg, 's5_m07_rerun')` → set `shard_dir = data/sprint5/s5_m07_rerun/processed/shards`.
4. Tapi `s5_m07_rerun` sendiri punya `stages: [train, eval]` dan `reuse_data_from: s5_00_lock_baseline` — **tidak punya data di direktori isolated-nya sendiri**.
5. `eval_metrics.py` line 627 mencoba `np.load('data/sprint5/s5_m07_rerun/processed/cic_val.npz')` → FileNotFoundError.

**Cascade:**
- Stage 1: 0/7 valid → `inconclusive`.
- Stage 3 (`s5_h01`, `s5_h02`): `skipped_inconclusive` — butuh `template_from_best_stage=stage1`.
- Stage 4 (`s5_c01`, `s5_c02`): `skipped_gate` — gate_pass=False.

**Fix yang dibutuhkan di `scripts/sprint5/research_runner.py`:**
Fungsi `apply_data_reuse_inputs` (atau resolver `reuse_data_from_best_stage`) harus **mengikuti chain `reuse_data_from`** dari run yang dipilih sebagai best. Jika best-stage-run itu sendiri memakai reuse data dari run lain, gunakan sumber aslinya.

**Workaround sementara:** Ubah stage1 runs di `run_registry.yaml` dari `reuse_data_from_best_stage: stage0` menjadi `reuse_data_from: s5_00_lock_baseline` secara eksplisit (hard-code sumber data dari run yang punya preprocessing).

In [14]:
# Cell: Diagnosis helper — validasi apakah data dir stage0 runs tersedia
# Jalankan ini setelah stage0 selesai untuk memastikan stage1 tidak akan crash
import os
from pathlib import Path

ROOT = Path('/content/nids-cnn-lstm-autoencoder') if Path('/content/nids-cnn-lstm-autoencoder').exists() else Path('.')

stage0_runs = ['s5_00_lock_baseline', 's5_m07_rerun']
data_fields_to_check = ['cic_val.npz', 'cic_test.npz', 'cse_test.npz']

print('=== Stage 0 Data Directory Diagnosis ===')
for run_id in stage0_runs:
    data_dir = ROOT / 'data' / 'sprint5' / run_id / 'processed'
    print(f'\n[RUN] {run_id}')
    print(f'  data_dir: {data_dir}')
    print(f'  exists  : {data_dir.exists()}')
    if data_dir.exists():
        for fname in data_fields_to_check:
            fpath = data_dir / fname
            status = f'OK ({fpath.stat().st_size // 1024} KB)' if fpath.exists() else 'MISSING'
            print(f'  {fname}: {status}')
        # Also check shards
        shard_dir = data_dir / 'shards'
        print(f'  shards/: {"exists" if shard_dir.exists() else "MISSING"}')
        if shard_dir.exists():
            manifests = list(shard_dir.rglob('manifest.json'))
            print(f'  manifests found: {len(manifests)}')
            for m in manifests:
                print(f'    -> {m.relative_to(shard_dir)}')
    else:
        print('  [WARN] No preprocessed data in isolated dir — run uses reuse_data_from')

print()
print('=== Recommendation for Stage 1 ===')
lock_data = ROOT / 'data' / 'sprint5' / 's5_00_lock_baseline' / 'processed'
if lock_data.exists():
    print(f'[OK] Use reuse_data_from: s5_00_lock_baseline')
    print(f'     Path valid: {lock_data}')
else:
    print('[WARN] s5_00_lock_baseline data not found. Stage0 must run first.')


=== Stage 0 Data Directory Diagnosis ===

[RUN] s5_00_lock_baseline
  data_dir: /content/nids-cnn-lstm-autoencoder/data/sprint5/s5_00_lock_baseline/processed
  exists  : True
  cic_val.npz: MISSING
  cic_test.npz: MISSING
  cse_test.npz: MISSING
  shards/: exists
  manifests found: 5
    -> cic/train/manifest.json
    -> cic/val/manifest.json
    -> cic/calib/manifest.json
    -> cic/test/manifest.json
    -> cse/test/manifest.json

[RUN] s5_m07_rerun
  data_dir: /content/nids-cnn-lstm-autoencoder/data/sprint5/s5_m07_rerun/processed
  exists  : False
  [WARN] No preprocessed data in isolated dir — run uses reuse_data_from

=== Recommendation for Stage 1 ===
[OK] Use reuse_data_from: s5_00_lock_baseline
     Path valid: /content/nids-cnn-lstm-autoencoder/data/sprint5/s5_00_lock_baseline/processed


### Temuan Tambahan

#### 1. `sprint_history_summary.count_under_guardrail = 0`
Kedua run stage0 punya **CSE FPR ~0.224**, jauh di atas guardrail CIC FPR cap 0.10. Karena tidak ada run yang lolos guardrail, `best_recall_under_guardrail = null` dan `adaptive_recall_target` hanya bisa pakai `fixed_floor_only = 0.75` (tidak ada baseline yang bisa dijumlahkan). Ini artinya **threshold `source_calib_guardrail_fpr010` sangat agresif menekan CIC FPR tapi justru menaikkan CSE FPR**.

#### 2. Bugs di Notebook Ini (sudah teridentifikasi)
- `run_stage()` memakai prefix `'Sprint4 {stage_name}'` → seharusnya `'Sprint5 {stage_name}'`.
- Cell 17 `report_path` menunjuk ke `RESEARCH_REPORT_CSE_F1_SPRINT4.md` → seharusnya `SPRINT5.md`.
- Kedua bugs anotasi saja, tidak mempengaruhi eksekusi stage.

#### 3. Signal Positif — mse_mae_mix Loss (m07)
- Wall time lebih cepat (~2m lebih singkat dari lock_baseline).
- CSE recall +7.2pp, CSE F1 +5.2pp, F1 gap turun drastis (0.103 → 0.046).
- **Implikasi:** Jika stage1 bisa dijalankan (bug fixed), maka model m07 adalah kandidat kuat sebagai base untuk target-threshold experiments.

#### 4. Kondisi Gate Sprint 5
Untuk gate_pass=True dibutuhkan CSE recall ≥ 0.75 di stage3. Stage0 sudah menunjukkan recall 0.698 (m07) — **gap ke target hanya 0.052**. Jika `target_percentile` (p95–p99) bisa menurunkan threshold dan mendorong recall ≥ 0.75 tanpa collapse (precision ≥ 0.30), gate bisa tercapai.

---

### Action Items untuk Re-Run

| Prioritas | Item | File | Detail |
|---|---|---|---|
| 🔴 P0 | Fix `reuse_data_from_best_stage` chain resolution | `scripts/sprint5/research_runner.py` | Ikuti `reuse_data_from` dari run terpilih |
| 🟡 P1 | Workaround: hardcode stage1 → `reuse_data_from: s5_00_lock_baseline` | `research/sprint5/run_registry.yaml` | Semua 7 run stage1 |
| 🟢 P2 | Fix label notebook `Sprint4 →  Sprint5` | `notebooks/sprint5_colab_runner.ipynb` | Anotasi saja |
| 🟢 P2 | Fix `report_path` `SPRINT4.md → SPRINT5.md` | `notebooks/sprint5_colab_runner.ipynb` | Cell 17 |